# 03 · Results

**Question for this notebook:** what changes after a goal, and how much of it survives scrutiny?

1. Checks first
2. The main estimate
3. Why the estimator adjusts for the before-shot level
4. Robustness: control group and window length

Needs `make all` (or `make estimate`) to have run for this config. Sections 3 and 4 re-estimate in the notebook, using the same functions as the pipeline.

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from confidence_shocks.config import ROOT, load_config

# Which run to look at. Switch to leagues_2015_16.yaml for the main analysis.
CONFIG = ROOT / "configs" / "default.yaml"
cfg = load_config(CONFIG)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")
pd.set_option("display.max_columns", 40)
print(cfg.name, "|", ", ".join(f"{c['name']} {c['season']}" for c in cfg.competitions))

In [ ]:
from confidence_shocks import estimate as est
from confidence_shocks.viz.plots import forest

shocks = pd.read_parquet(cfg.shocks_path)
out = cfg.out_dir
checks = pd.read_csv(out / "checks.csv")
results = pd.read_csv(out / "results.csv")
results_shooter = pd.read_csv(out / "results_unadjusted.csv")
balance = pd.read_csv(out / "balance.csv")

## 1 · Checks first

- **balance:** score difference and match minute are not adjusted for, so after xG weighting they must already match (standardised difference at most `max_smd`).
- **placebo:** the same estimator, moved one window back, where scoring cannot have had any effect yet. A failure means goals were not a pure coin flip given xG and recent form, so read the results with that caveat.

In [ ]:
print("ALL CHECKS PASSED" if checks["passed"].all() else f"{(~checks['passed']).sum()} CHECKS FAILED")
checks.round(3)

## 2 · The main estimate

In [ ]:
cols = ["metric", "kind", "effect", "lo", "hi", "effect_sd", "p", "q", "n_goal", "n_control"]
results[cols].round(3)

In [ ]:
fig = forest({"shooter only": results_shooter, "minus teammates": results},
             f"{cfg.name}: after scoring vs after a comparable saved shot ({cfg.method})")
fig

Effects are in standard deviations of the after-shot level. A finding needs `q < 0.05`; anything else is "no clear effect at this sample size". Choice metrics are measured more reliably than execution metrics, so a null on `pass_cmp` says less than a null on a choice metric.

## 3 · Why adjust for the before-shot level

The first version used after-minus-before changes. When scorers start at a different level, a change score picks that gap up as regression to the mean: the "effect" on each metric points the opposite way to the before-shot gap. Plotting the gap against the effect shows whether that is happening; under `ancova` the relationship should weaken.

In [ ]:
kwargs = dict(controls=cfg.controls, n_boot=1000, seed=cfg.seed)
change = est.estimate_all(shocks, method="change", **kwargs)
ancova = est.estimate_all(shocks, method="ancova", **kwargs)
gap = balance.set_index("quantity").loc[[f"{m}_pre" for m in change["metric"]], "effect_sd"].to_numpy()

fig, ax = plt.subplots(figsize=(6, 4))
for table, label in [(change, "after minus before"), (ancova, "ancova")]:
    r = np.corrcoef(gap, table["effect_sd"])[0, 1]
    ax.scatter(gap, table["effect_sd"], label=f"{label} (r = {r:.2f})")
for x, y, name in zip(gap, change["effect_sd"], change["metric"]):
    ax.annotate(name, (x, y), fontsize=7, alpha=0.7)
ax.axhline(0, color="grey", linewidth=0.5)
ax.axvline(0, color="grey", linewidth=0.5)
ax.set_xlabel("before-shot gap, goals minus controls (SD)")
ax.set_ylabel("estimated effect of scoring (SD)")
ax.legend()
plt.show()

In [ ]:
forest({"after minus before": change, "ancova": ancova}, "Same data, two estimators (minus teammates)")

## 4 · Robustness

### Control group

Who a goal is compared with: every non-goal (`all`), saved shots (`on_target`, the default), shots off the post (`post`, the cleanest lottery but few of them), or saved plus post (`near_miss`).

In [ ]:
METRICS_SHOWN = ["receipts", "dribbles", "pass_length", "pass_cmp"]
rows = []
for controls in est.CONTROL_SETS:
    table = est.estimate_all(shocks, controls=controls, method=cfg.method, n_boot=1000, seed=cfg.seed)
    table.insert(0, "controls", controls)
    rows.append(table[table["metric"].isin(METRICS_SHOWN)])
by_controls = pd.concat(rows)
by_controls.pivot(index="metric", columns="controls", values="effect_sd").round(2)

In [ ]:
by_controls.pivot(index="metric", columns="controls", values="n_control")

### Window length

Rebuilds the shock table for each window from the downloaded events (no new downloads). Takes a minute or two on the league data.

In [ ]:
from confidence_shocks.shocks import match_shocks

def shocks_for_window(window):
    tables = []
    for match_id in shocks["match_id"].unique():
        events = pd.read_parquet(cfg.events_dir / f"{match_id}.parquet")
        spans = pd.read_parquet(cfg.spans_dir / f"{match_id}.parquet")
        tables.append(match_shocks(events, spans, window))
    return pd.concat(tables, ignore_index=True)

rows = []
for window in (5, 10, 15, 20):
    table = shocks if window == cfg.window_minutes else shocks_for_window(window)
    res = est.estimate_all(table, controls=cfg.controls, method=cfg.method, n_boot=1000, seed=cfg.seed)
    res.insert(0, "window", window)
    rows.append(res[res["metric"].isin(METRICS_SHOWN)])
by_window = pd.concat(rows)
by_window.pivot(index="metric", columns="window", values="effect_sd").round(2)

A real effect should keep its sign across windows and control groups, and shrink smoothly rather than flip. Report these tables whatever they show (the analysis plan commits to that).